In [ ]:
#@title Configure { display-mode: "form" }

#@markdown Folder **in Drive** holding the two archives:
ARCHIVE_DIR = "/content/drive/MyDrive/DriveJEPA"  #@param {type:"string"}

#@markdown Where to extract (Colab local disk — fast, wiped on disconnect):
DATAROOT = "/content/nuscenes"  #@param {type:"string"}

VERSION = "v1.0-mini"  #@param ["v1.0-mini", "v1.0-trainval"]

#@markdown Untick if you already extracted in this session:
DO_EXTRACT = True  #@param {type:"boolean"}

import os, time
from google.colab import drive
drive.mount('/content/drive')

print(f"\narchives : {ARCHIVE_DIR}")
if os.path.isdir(ARCHIVE_DIR):
    for f in sorted(os.listdir(ARCHIVE_DIR)):
        sz = os.path.getsize(os.path.join(ARCHIVE_DIR, f)) / 1e9
        print(f"           {f}  ({sz:.2f} GB)")
else:
    print("           NOT FOUND — fix ARCHIVE_DIR above")

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/DriveJEPA"))

In [ ]:
import os

dataset_dir = "/content/drive/MyDrive/DriveJEPA/nuScenes"
os.makedirs(dataset_dir, exist_ok=True)

print("Dataset folder ready:", dataset_dir)

In [ ]:
import os

file_path = "/content/drive/MyDrive/DriveJEPA/v1.0-mini.tgz"

print("File exists:", os.path.exists(file_path))
print("File size (GB):", round(os.path.getsize(file_path) / 1e9, 2))

In [ ]:
import os

drivejepa_dir = "/content/drive/MyDrive/DriveJEPA"

for filename in ["v1.0-mini.tgz", "can_bus.zip"]:
    path = os.path.join(drivejepa_dir, filename)
    print(f"{filename}:")
    print("  Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("  Size (MB):", round(os.path.getsize(path) / 1e6, 2))

In [ ]:
import os

extract_path = "/content/nuscenes"

if os.path.exists(extract_path):
    print("Extracted folders:", os.listdir(extract_path))
else:
    print("Dataset not extracted yet")

In [ ]:
#@title Extract to local disk  (~1–2 min)

if DO_EXTRACT:
    os.makedirs(DATAROOT, exist_ok=True)
    t0 = time.time()

    # Copy off Drive first — extracting straight from a Drive path is far slower.
    print("copying archives to local disk...")
    !cp "{ARCHIVE_DIR}/v1.0-mini.tgz" /content/ 2>/dev/null
    !cp "{ARCHIVE_DIR}/can_bus.zip"   /content/ 2>/dev/null

    print("extracting v1.0-mini.tgz...")
    !tar -xzf /content/v1.0-mini.tgz -C "{DATAROOT}"

    print("extracting can_bus.zip...")
    !unzip -q -o /content/can_bus.zip -d "{DATAROOT}"

    # Free ~5 GB — the archives are still in Drive.
    !rm -f /content/v1.0-mini.tgz /content/can_bus.zip
    print(f"\ndone in {time.time() - t0:.0f}s")

print("\ncontents of DATAROOT:")
for d in sorted(os.listdir(DATAROOT)):
    print(f"  {d}")

n_tables = len([f for f in os.listdir(f"{DATAROOT}/{VERSION}") if f.endswith('.json')])
print(f"\nmetadata tables: {n_tables}   (13 = correct base set; 2–3 = you grabbed an expansion)")

In [ ]:
#@title Extract to local disk  (~1–2 min)

if DO_EXTRACT:
    os.makedirs(DATAROOT, exist_ok=True)
    t0 = time.time()

    # Copy off Drive first — extracting straight from a Drive path is far slower.
    print("copying archives to local disk...")
    !cp "{ARCHIVE_DIR}/v1.0-mini.tgz" /content/ 2>/dev/null
    !cp "{ARCHIVE_DIR}/can_bus.zip"   /content/ 2>/dev/null

    print("extracting v1.0-mini.tgz...")
    !tar -xzf /content/v1.0-mini.tgz -C "{DATAROOT}"

    print("extracting can_bus.zip...")
    !unzip -q -o /content/can_bus.zip -d "{DATAROOT}"

    # Free ~5 GB — the archives are still in Drive.
    !rm -f /content/v1.0-mini.tgz /content/can_bus.zip
    print(f"\ndone in {time.time() - t0:.0f}s")

print("\ncontents of DATAROOT:")
for d in sorted(os.listdir(DATAROOT)):
    print(f"  {d}")

n_tables = len([f for f in os.listdir(f"{DATAROOT}/{VERSION}") if f.endswith('.json')])
print(f"\nmetadata tables: {n_tables}   (13 = correct base set; 2–3 = you grabbed an expansion)")

In [ ]:
#@title Install the devkit
!pip install -q nuscenes-devkit
print("ok")

In [ ]:
!pip install -q --no-deps nuscenes-devkit
!pip install -q pyquaternion cachetools descartes

In [ ]:
!pip install -q "numpy==1.26.4" "Shapely==2.0.7" fire parameterized

In [ ]:
import numpy as np
import shapely

print("NumPy:", np.__version__)
print("Shapely:", shapely.__version__)

from nuscenes.nuscenes import NuScenes
from nuscenes.can_bus.can_bus_api import NuScenesCanBus

print("nuScenes imports successful!")

In [ ]:
import os

DATAROOT = "/content/nuscenes"

print("Dataset exists:", os.path.isdir(DATAROOT))
print("Folders:", os.listdir(DATAROOT) if os.path.isdir(DATAROOT) else "Not found")

In [ ]:
VERSION = "v1.0-mini"
DATAROOT = "/content/nuscenes"

import numpy as np, shapely
from nuscenes.nuscenes import NuScenes
from nuscenes.can_bus.can_bus_api import NuScenesCanBus
print("numpy", np.__version__, "| shapely", shapely.__version__)

nusc = NuScenes(version=VERSION, dataroot=DATAROOT, verbose=False)
can  = NuScenesCanBus(dataroot=DATAROOT)
s = nusc.scene[0]
print("scenes", len(nusc.scene), "| samples", len(nusc.sample))
print("CAN msgs", len(can.get_messages(s["name"], "vehicle_monitor")))
print("PASS")

In [ ]:
#@title 1 · Footprint — files, not gigabytes

from collections import Counter

n_files, n_bytes, per_dir = 0, 0, Counter()
for dirpath, _, filenames in os.walk(DATAROOT):
    n_files += len(filenames)
    top = os.path.relpath(dirpath, DATAROOT).split(os.sep)[0]
    per_dir[top] += len(filenames)
    for f in filenames:
        try:
            n_bytes += os.path.getsize(os.path.join(dirpath, f))
        except OSError:
            pass

print(f"total files : {n_files:,}")
print(f"total size  : {n_bytes / 1e9:.2f} GB\n")
for d, c in per_dir.most_common():
    print(f"  {d:<16} {c:>8,} files")

print(f"\nScaling note: mini is 10 of 1000 scenes. Full trainval is roughly")
print(f"{n_files * 85:,.0f} files by this ratio — against a 250,000 file quota on")
print("HPRC scratch. Shard into tar/webdataset before moving past mini.")

In [ ]:
#@title 2 · Structure

from nuscenes.nuscenes import NuScenes
nusc = NuScenes(version=VERSION, dataroot=DATAROOT, verbose=False)

print(f"scenes      : {len(nusc.scene)}")
print(f"samples     : {len(nusc.sample)}        annotated keyframes @ 2 Hz")
print(f"sample_data : {len(nusc.sample_data):,}      every reading, sweeps included")
print(f"annotations : {len(nusc.sample_annotation):,}\n")

ch = Counter(
    nusc.get("sensor", nusc.get("calibrated_sensor",
             sd["calibrated_sensor_token"])["sensor_token"])["channel"]
    for sd in nusc.sample_data
)
for c, n in sorted(ch.items(), key=lambda kv: -kv[1]):
    print(f"  {c:<24} {n:>7,}")


In [ ]:
#@title 3 · One scene

scene = nusc.scene[0]
first = nusc.get("sample", scene["first_sample_token"])
last  = nusc.get("sample", scene["last_sample_token"])
dur   = (last["timestamp"] - first["timestamp"]) / 1e6

print(f"name        : {scene['name']}")
print(f"description : {scene['description']}")
print(f"keyframes   : {scene['nbr_samples']}")
print(f"duration    : {dur:.1f}s  ({scene['nbr_samples']/dur:.2f} Hz)\n")
for c in first["data"]:
    print(f"  {c}")

In [ ]:
#@title 4 · The six cameras

import matplotlib.pyplot as plt
from PIL import Image

order = ["CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
         "CAM_BACK_LEFT",  "CAM_BACK",  "CAM_BACK_RIGHT"]
fig, axes = plt.subplots(2, 3, figsize=(16, 6))
for ax, cam in zip(axes.ravel(), order):
    sd = nusc.get("sample_data", first["data"][cam])
    ax.imshow(Image.open(os.path.join(DATAROOT, sd["filename"])))
    ax.set_title(cam, fontsize=9)
    ax.axis("off")
plt.suptitle(f"{scene['name']} — keyframe 0", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
#@title 5 · CAN bus — action and state

import numpy as np
from nuscenes.can_bus.can_bus_api import NuScenesCanBus

can = NuScenesCanBus(dataroot=DATAROOT)
blacklist = set(f"scene-{n:04d}" for n in getattr(can, "can_blacklist", []))
print(f"scenes with unusable CAN data: {sorted(blacklist) or 'none'}\n")

assert scene["name"] not in blacklist, "pick a different scene"

for msg in ["vehicle_monitor", "steeranglefeedback", "pose", "ms_imu"]:
    try:
        rows = can.get_messages(scene["name"], msg)
    except Exception as e:
        print(f"{msg:<22} unavailable ({type(e).__name__})")
        continue
    ts = np.array([r["utime"] for r in rows], dtype=np.int64)
    hz = len(rows) / ((ts[-1] - ts[0]) / 1e6)
    print(f"{msg:<22} {len(rows):>6} msgs   {hz:>6.1f} Hz")
    print(f"    {', '.join(sorted(rows[0].keys()))}\n")

In [ ]:
#@title 6 · Alignment — the decision to settle now

key_ts, tok = [], scene["first_sample_token"]
while tok:
    s = nusc.get("sample", tok)
    key_ts.append(s["timestamp"])
    tok = s["next"]
key_ts = np.array(key_ts, dtype=np.int64)

fig, ax = plt.subplots(figsize=(11, 4))
for msg in ["vehicle_monitor", "steeranglefeedback", "pose"]:
    try:
        rows = can.get_messages(scene["name"], msg)
    except Exception:
        continue
    can_ts = np.array([r["utime"] for r in rows], dtype=np.int64)
    idx = np.searchsorted(can_ts, key_ts).clip(0, len(can_ts) - 1)
    gap = np.abs(can_ts[idx] - key_ts) / 1e3
    print(f"{msg:<22} median {np.median(gap):6.1f} ms   "
          f"p95 {np.percentile(gap, 95):6.1f} ms   max {gap.max():6.1f} ms")
    ax.plot(gap, marker="o", ms=3, lw=1, label=msg)

ax.axhline(20, ls="--", lw=1, color="grey")
ax.text(0.5, 22, "20 ms", fontsize=8, color="grey")
ax.set_xlabel("keyframe index")
ax.set_ylabel("gap to nearest CAN reading (ms)")
ax.set_title(f"CAN-to-keyframe alignment — {scene['name']}")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("\nUnder ~20 ms: nearest-neighbour is defensible.")
print("Above it: interpolate, or resample onto a fixed grid.")
print("Record whichever you pick — it changes what the model learns.")

In [ ]:
#@title 7 · Build the per-keyframe (action, state) table

import pandas as pd

vm = pd.DataFrame(can.get_messages(scene["name"], "vehicle_monitor")).sort_values("utime")
print("vehicle_monitor columns:\n ", list(vm.columns), "\n")

joined = pd.merge_asof(
    pd.DataFrame({"utime": key_ts}).sort_values("utime"),
    vm, on="utime", direction="nearest",
)
display(joined.head(12))
print(f"shape: {joined.shape}  — one row per keyframe, ready to condition on.")

# The columns that matter for an action-conditioned world model.
ACTION = [c for c in ["steering", "throttle", "brake"] if c in joined.columns]
STATE  = [c for c in ["vehicle_speed", "yaw_rate"]      if c in joined.columns]
print(f"\naction columns found: {ACTION}")
print(f"state  columns found: {STATE}")

In [ ]:
#@title 8 · Sanity-plot the control signals over one scene

cols = ACTION + STATE
if cols:
    t = (joined["utime"] - joined["utime"].iloc[0]) / 1e6
    fig, axes = plt.subplots(len(cols), 1, figsize=(11, 1.8 * len(cols)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, cols):
        ax.plot(t, joined[c], lw=1.2)
        ax.set_ylabel(c, fontsize=9)
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel("seconds into scene")
    plt.suptitle(f"Control signals — {scene['name']}", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("No expected columns found — check the printout in cell 7 and adjust.")

In [ ]:
from nuscenes.can_bus.can_bus_api import NuScenesCanBus

can = NuScenesCanBus(dataroot=DATAROOT)

for scene in nusc.scene:
    name = scene["name"]

    try:
        vm = can.get_messages(name, "vehicle_monitor")
        sa = can.get_messages(name, "steeranglefeedback")

        print(
            f"{name}: "
            f"vehicle_monitor={len(vm)}, "
            f"steeranglefeedback={len(sa)}"
        )

    except Exception as e:
        print(f"{name}: ERROR - {e}")

In [ ]:
import numpy as np
from PIL import Image
import os

CAMS = ["CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT"]
H, W = 72, 128

scene = nusc.scene[0]
samples = []
token = scene["first_sample_token"]

while token:
    sample = nusc.get("sample", token)
    samples.append(sample)
    token = sample["next"]

frames = np.zeros((len(samples), 9, H, W), dtype=np.uint8)
timestamps = np.zeros(len(samples), dtype=np.int64)

for i, sample in enumerate(samples):
    timestamps[i] = sample["timestamp"]

    for j, camera in enumerate(CAMS):
        sd = nusc.get("sample_data", sample["data"][camera])
        image_path = os.path.join(DATAROOT, sd["filename"])

        with Image.open(image_path) as img:
            img = img.convert("RGB").resize((W, H), Image.Resampling.BILINEAR)
            frames[i, j*3:(j+1)*3] = np.asarray(img).transpose(2, 0, 1)

print("Scene:", scene["name"])
print("Frames shape:", frames.shape)
print("Data type:", frames.dtype)
print("Timestamps shape:", timestamps.shape)

In [ ]:
import pandas as pd
import numpy as np

scene_name = scene["name"]

vm = pd.DataFrame(can.get_messages(scene_name, "vehicle_monitor"))
sa = pd.DataFrame(can.get_messages(scene_name, "steeranglefeedback"))

# Keep the original CAN timestamps to measure alignment gaps.
vm = vm[["utime", "throttle", "brake"]].sort_values("utime")
sa = sa[["utime", "value"]].rename(
    columns={"value": "steer"}
).sort_values("utime")

vm = vm.rename(columns={"utime": "vm_time"})
sa = sa.rename(columns={"utime": "steer_time"})

keys = pd.DataFrame({"timestamp": timestamps}).sort_values("timestamp")

# Match each camera keyframe to the nearest CAN reading.
aligned = pd.merge_asof(
    keys, vm,
    left_on="timestamp",
    right_on="vm_time",
    direction="nearest"
)

aligned = pd.merge_asof(
    aligned, sa,
    left_on="timestamp",
    right_on="steer_time",
    direction="nearest"
)

# nuScenes timestamps are in microseconds.
aligned["gap_vm_ms"] = (
    (aligned["timestamp"] - aligned["vm_time"]).abs() / 1000
)

aligned["gap_steer_ms"] = (
    (aligned["timestamp"] - aligned["steer_time"]).abs() / 1000
)

print("Scene:", scene_name)
print("Aligned rows:", len(aligned))

print(aligned[
    ["throttle", "brake", "steer", "gap_vm_ms", "gap_steer_ms"]
].head())

print("\nVehicle monitor median gap:",
      round(aligned["gap_vm_ms"].median(), 1), "ms")

print("Vehicle monitor p95 gap:",
      round(aligned["gap_vm_ms"].quantile(0.95), 1), "ms")

print("Rows with vehicle monitor gap >250 ms:",
      int((aligned["gap_vm_ms"] > 250).sum()))

In [ ]:
print("Vehicle monitor gaps:")
print(aligned["gap_vm_ms"].describe())

print("\nSteering gaps:")
print(aligned["gap_steer_ms"].describe())

print("\nRows with vehicle monitor gaps >250 ms:")
print(
    aligned.loc[
        aligned["gap_vm_ms"] > 250,
        ["timestamp", "vm_time", "gap_vm_ms", "throttle", "brake"]
    ].to_string(index=False)
)

In [ ]:
print("Camera timestamp range:")
print(timestamps.min(), timestamps.max())

print("\nVehicle monitor timestamp range:")
print(vm["vm_time"].min(), vm["vm_time"].max())

print("\nSteering timestamp range:")
print(sa["steer_time"].min(), sa["steer_time"].max())

print("\nVehicle monitor interval statistics (ms):")
print((vm["vm_time"].diff().dropna() / 1000).describe())


### Timestamp Range and CAN Bus Frequency Check

I checked the timestamp ranges of the camera frames, vehicle monitor, and steering measurements for `scene-0061` to understand their time coverage.

**Observations:**
- Camera timestamps range from `1532402927647951` to `1532402946797517`.
- Vehicle monitor timestamps range from `1532402928127800` to `1532402947127060`.
- Steering timestamps range from `1532402927656536` to `1532402946801204`.
- Vehicle monitor messages are usually recorded about **500 ms apart**, which is approximately **2 Hz**.
- The maximum interval between vehicle monitor messages is approximately **1,000 ms**.

The vehicle monitor starts slightly after the first camera frame, which explains the larger timestamp gap for that frame. The longer interval between two vehicle monitor messages may also contribute to alignment gaps.

Overall, the timestamp ranges overlap for most of the scene. We will keep the alignment gaps in the processed dataset so that frames with larger differences can be identified during validation.


In [ ]:
# Flag camera frames based on CAN bus timestamp gaps.
# 250 ms is a provisional threshold for checking alignment quality.

MAX_GAP_MS = 250

aligned["vm_valid"] = aligned["gap_vm_ms"] <= MAX_GAP_MS
aligned["steer_valid"] = aligned["gap_steer_ms"] <= MAX_GAP_MS

# A frame passes this check only if both CAN matches
# are within the selected threshold.
aligned["action_valid"] = (
    aligned["vm_valid"] & aligned["steer_valid"]
)

print("Scene:", scene_name)
print("Total keyframes:", len(aligned))
print("Frames passing alignment check:", aligned["action_valid"].sum())
print("Frames flagged for review:", (~aligned["action_valid"]).sum())

print("\nAlignment quality:")
print(
    aligned[
        ["gap_vm_ms", "gap_steer_ms", "action_valid"]
    ].describe(include="all")
)

In [ ]:
# Extract the three front-facing cameras for every Mini scene.

scene_data = {}

for current_scene in nusc.scene:
    scene_samples = []
    token = current_scene["first_sample_token"]

    while token:
        sample = nusc.get("sample", token)
        scene_samples.append(sample)
        token = sample["next"]

    scene_frames = np.zeros(
        (len(scene_samples), 9, H, W),
        dtype=np.uint8
    )

    scene_times = np.zeros(
        len(scene_samples),
        dtype=np.int64
    )

    for i, sample in enumerate(scene_samples):
        scene_times[i] = sample["timestamp"]

        for j, camera in enumerate(CAMS):
            sd = nusc.get(
                "sample_data",
                sample["data"][camera]
            )

            image_path = os.path.join(
                DATAROOT,
                sd["filename"]
            )

            with Image.open(image_path) as img:
                img = img.convert("RGB")
                img = img.resize(
                    (W, H),
                    Image.Resampling.BILINEAR
                )

                scene_frames[i, j*3:(j+1)*3] = (
                    np.asarray(img).transpose(2, 0, 1)
                )

    scene_data[current_scene["name"]] = {
        "frames": scene_frames,
        "timestamps": scene_times
    }

    print(
        current_scene["name"],
        "| frames:",
        scene_frames.shape
    )

print("\nTotal scenes:", len(scene_data))
print(
    "Total keyframes:",
    sum(len(s["timestamps"]) for s in scene_data.values())
)


### Extracting Camera Frames from All 10 Scenes

After checking the extraction process on one scene, I used the same approach for all 10 scenes in nuScenes Mini.

For each keyframe, I extracted images from the front-left, front, and front-right cameras. I resized them to 128 × 72 pixels and combined the RGB images into a 9-channel array.

The extraction completed for all 10 scenes, giving **404 keyframes** in total. Each scene's frames and timestamps are stored separately in `scene_data`.


In [ ]:
# Align throttle, brake, and steering for all 10 scenes.

aligned_data = {}

for name, data in scene_data.items():

    vm = pd.DataFrame(
        can.get_messages(name, "vehicle_monitor")
    )[["utime", "throttle", "brake"]].sort_values("utime")

    sa = pd.DataFrame(
        can.get_messages(name, "steeranglefeedback")
    )[["utime", "value"]].sort_values("utime")

    vm = vm.rename(columns={"utime": "vm_time"})
    sa = sa.rename(columns={
        "utime": "steer_time",
        "value": "steer"
    })

    keys = pd.DataFrame({
        "timestamp": data["timestamps"]
    }).sort_values("timestamp")

    matched = pd.merge_asof(
        keys, vm,
        left_on="timestamp",
        right_on="vm_time",
        direction="nearest"
    )

    matched = pd.merge_asof(
        matched, sa,
        left_on="timestamp",
        right_on="steer_time",
        direction="nearest"
    )

    matched["gap_vm_ms"] = (
        (matched["timestamp"] - matched["vm_time"]).abs() / 1000
    )

    matched["gap_steer_ms"] = (
        (matched["timestamp"] - matched["steer_time"]).abs() / 1000
    )

    matched["action_valid"] = (
        (matched["gap_vm_ms"] <= 250) &
        (matched["gap_steer_ms"] <= 250)
    )

    aligned_data[name] = matched

    print(
        f"{name}: "
        f"{len(matched)} frames | "
        f"{matched['action_valid'].sum()} passed | "
        f"{(~matched['action_valid']).sum()} flagged"
    )

print("\nTotal aligned frames:",
      sum(len(df) for df in aligned_data.values()))

print("Total flagged frames:",
      sum((~df["action_valid"]).sum()
          for df in aligned_data.values()))


### Aligning Driving Actions with Camera Frames

Next, I matched the camera frames with throttle, brake, and steering measurements for all 10 scenes.

I used `vehicle_monitor` for throttle and brake, and `steeranglefeedback` for steering. Since these measurements have different timestamps, I used nearest-neighbor matching to find the closest CAN bus reading for each camera frame.

I also calculated the timestamp gaps and flagged frames where either gap was greater than 250 ms.

Out of **404 frames, 397 passed the initial alignment check and 7 were flagged**. I kept all the frames for further review.


In [ ]:
# Summarize timestamp alignment quality across all scenes.

summary = []

for name, df in aligned_data.items():
    summary.append({
        "scene": name,
        "frames": len(df),
        "vm_median_ms": round(df["gap_vm_ms"].median(), 1),
        "vm_p95_ms": round(df["gap_vm_ms"].quantile(0.95), 1),
        "vm_max_ms": round(df["gap_vm_ms"].max(), 1),
        "steer_median_ms": round(df["gap_steer_ms"].median(), 1),
        "steer_max_ms": round(df["gap_steer_ms"].max(), 1),
        "flagged": int((~df["action_valid"]).sum())
    })

summary_df = pd.DataFrame(summary)

print(summary_df.to_string(index=False))

print("\nOverall vehicle monitor median gap:",
      round(pd.concat(
          [df["gap_vm_ms"] for df in aligned_data.values()]
      ).median(), 1), "ms")

print("Overall steering median gap:",
      round(pd.concat(
          [df["gap_steer_ms"] for df in aligned_data.values()]
      ).median(), 1), "ms")


### Checking Alignment Quality Across Scenes

I calculated the median, 95th percentile, and maximum timestamp gaps for each scene to see how well the CAN bus measurements aligned with the camera frames.

The overall median gap was **171.0 ms for throttle and brake** and **2.4 ms for steering**.

Steering measurements were generally closer to the camera timestamps. I also noticed that `scene-0655` had a vehicle monitor median gap of 242.6 ms, which was close to the 250 ms threshold.

This check helped identify differences between scenes and confirmed why the actual timestamp gaps should be kept in the processed data.


In [ ]:
import os
import numpy as np

OUTPUT_DIR = "/content/drive/MyDrive/DriveJEPA/nuscenes_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name, data in scene_data.items():
    df = aligned_data[name]

    # Three driving actions in a fixed order:
    # throttle, brake, steering
    actions = df[
        ["throttle", "brake", "steer"]
    ].to_numpy(dtype=np.float32)

    output_path = os.path.join(OUTPUT_DIR, f"{name}.npz")

    np.savez_compressed(
        output_path,
        frames=data["frames"],
        timestamps=data["timestamps"],
        actions=actions,
        gap_vm_ms=df["gap_vm_ms"].to_numpy(dtype=np.float32),
        gap_steer_ms=df["gap_steer_ms"].to_numpy(dtype=np.float32),
        action_valid=df["action_valid"].to_numpy(dtype=bool)
    )

    print(
        f"Saved {name}: "
        f"{len(data['frames'])} frames | "
        f"{actions.shape[0]} actions"
    )

print("\nSaved files:", len([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".npz")
]))
print("Output folder:", OUTPUT_DIR)

In [ ]:
# Verify all saved scene files before using them for training.

import os
import numpy as np

total_frames = 0
total_flagged = 0

for name in scene_data:
    path = os.path.join(OUTPUT_DIR, f"{name}.npz")

    with np.load(path) as data:
        frames = data["frames"]
        actions = data["actions"]
        timestamps = data["timestamps"]
        valid = data["action_valid"]

        n = len(frames)

        assert frames.shape == (n, 9, 72, 128)
        assert frames.dtype == np.uint8
        assert actions.shape == (n, 3)
        assert len(timestamps) == n
        assert len(valid) == n
        assert len(data["gap_vm_ms"]) == n
        assert len(data["gap_steer_ms"]) == n
        assert np.isfinite(actions).all()

        flagged = int((~valid).sum())
        total_frames += n
        total_flagged += flagged

        print(f"{name}: VERIFIED | {n} frames | {flagged} flagged")

print("\nTotal verified frames:", total_frames)
print("Total flagged frames:", total_flagged)

assert total_frames == 404
assert total_flagged == 7

print("\nAll saved files passed verification!")

In [ ]:
import json
import numpy as np
import os

# Create a reproducible scene-level split.
scene_names = sorted(scene_data.keys())

rng = np.random.default_rng(42)
shuffled_scenes = rng.permutation(scene_names).tolist()

train_scenes = sorted(shuffled_scenes[:7])
val_scenes = sorted(shuffled_scenes[7:])

split = {
    "train": train_scenes,
    "val": val_scenes
}

# Save the split alongside the processed scene files.
split_path = os.path.join(OUTPUT_DIR, "scene_split.json")

with open(split_path, "w") as f:
    json.dump(split, f, indent=2)

print("Training scenes:", train_scenes)
print("Validation scenes:", val_scenes)

print("\nTraining frames:",
      sum(len(scene_data[s]["timestamps"]) for s in train_scenes))

print("Validation frames:",
      sum(len(scene_data[s]["timestamps"]) for s in val_scenes))

print("\nSplit saved to:", split_path)

assert len(train_scenes) == 7
assert len(val_scenes) == 3
assert set(train_scenes).isdisjoint(val_scenes)
assert set(train_scenes + val_scenes) == set(scene_names)

print("Scene-level split verified!")

In [ ]:
import os
import json

OUTPUT_DIR = "/content/drive/MyDrive/DriveJEPA/nuscenes_processed"
split_path = os.path.join(OUTPUT_DIR, "scene_split.json")

# Read the existing split.
with open(split_path, "r") as f:
    old_split = json.load(f)

# Convert the existing train/val lists into a scene-level mapping.
if "train" in old_split and "val" in old_split:
    train_scenes = old_split["train"]
    val_scenes = old_split["val"]

    assert len(train_scenes) == 7
    assert len(val_scenes) == 3
    assert set(train_scenes).isdisjoint(val_scenes)

    new_split = {
        scene: {"split": "train"}
        for scene in train_scenes
    }

    new_split.update({
        scene: {"split": "val"}
        for scene in val_scenes
    })

else:
    # The manifest has already been converted.
    new_split = old_split

assert len(new_split) == 10
assert sum(v["split"] == "train" for v in new_split.values()) == 7
assert sum(v["split"] == "val" for v in new_split.values()) == 3

# Save the corrected manifest.
with open(split_path, "w") as f:
    json.dump(new_split, f, indent=4)

print("Manifest updated successfully!")
print("Training scenes: 7")
print("Validation scenes: 3")
print("Saved to:", split_path)


### Training and Validation Split

The 10 processed nuScenes Mini scenes were divided into 7 training scenes and 3 validation scenes using a fixed random seed of 42.

The split was performed at the scene level to prevent frames from the same driving sequence from appearing in both datasets.

- Training: 7 scenes, 284 keyframes
- Validation: 3 scenes, 120 keyframes
- Total: 404 keyframes

The split was verified to ensure that no scenes overlap between training and validation. The scene names were saved in `scene_split.json` for reproducibility.

All frames were retained, including those flagged during timestamp alignment. The corresponding quality flags remain available in the processed `.npz` files.


In [ ]:
import os
import json
import numpy as np

OUTPUT_DIR = "/content/drive/MyDrive/DriveJEPA/nuscenes_processed"

# Load the saved scene-level split manifest.
with open(os.path.join(OUTPUT_DIR, "scene_split.json"), "r") as f:
    split = json.load(f)

# Separate scenes into training and validation groups.
train_scenes = [
    scene for scene, info in split.items()
    if info["split"] == "train"
]

val_scenes = [
    scene for scene, info in split.items()
    if info["split"] == "val"
]

scene_groups = {
    "train": train_scenes,
    "val": val_scenes
}

# Verify the scene-level split.
assert len(split) == 10
assert len(train_scenes) == 7
assert len(val_scenes) == 3
assert set(train_scenes).isdisjoint(set(val_scenes))
assert set(info["split"] for info in split.values()) == {"train", "val"}

total_frames = 0
total_flagged = 0

# Check every saved scene file.
for group in ["train", "val"]:

    group_frames = 0
    group_flagged = 0

    print(f"\n{group.upper()} DATASET")

    for scene_name in scene_groups[group]:

        file_path = os.path.join(
            OUTPUT_DIR, f"{scene_name}.npz"
        )

        assert os.path.isfile(file_path), (
            f"Missing file: {scene_name}"
        )

        with np.load(file_path) as data:

            n = len(data["frames"])

            # Verify saved frame and action dimensions.
            assert data["frames"].shape == (n, 9, 72, 128)
            assert data["actions"].shape == (n, 3)
            assert len(data["timestamps"]) == n
            assert len(data["action_valid"]) == n

            # Check that action values are finite.
            assert np.isfinite(data["actions"]).all()

            # Count frames with alignment gaps above the threshold.
            flagged = int(
                (~data["action_valid"]).sum()
            )

            group_frames += n
            group_flagged += flagged

            print(
                f"{scene_name}: {n} frames | "
                f"{flagged} flagged"
            )

    total_frames += group_frames
    total_flagged += group_flagged

    print(f"{group} total: {group_frames} frames")
    print(f"{group} flagged: {group_flagged}")

# Final dataset checks.
assert total_frames == 404
assert total_flagged == 7

print("\nTotal frames:", total_frames)
print("Total flagged:", total_flagged)
print("Final dataset handoff check: PASSED")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

scene = nusc.scene[0]
sample = nusc.get("sample", scene["first_sample_token"])

cameras = [
    "CAM_FRONT_LEFT",
    "CAM_FRONT",
    "CAM_FRONT_RIGHT"
]

fig, axes = plt.subplots(3, 1, figsize=(18, 18))

for ax, camera in zip(axes, cameras):
    sd = nusc.get("sample_data", sample["data"][camera])
    image_path = os.path.join(DATAROOT, sd["filename"])

    with Image.open(image_path) as img:
        ax.imshow(img.copy())

    ax.set_title(camera, fontsize=16)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

scene_name = "scene-0061"
file_path = os.path.join(OUTPUT_DIR, f"{scene_name}.npz")

with np.load(file_path) as data:
    timestamps = data["timestamps"]
    actions = data["actions"]
    gap_vm_ms = data["gap_vm_ms"]
    gap_steer_ms = data["gap_steer_ms"]
    action_valid = data["action_valid"]

# Time in seconds from the first camera keyframe.
time_s = (timestamps - timestamps[0]) / 1e6

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

signals = [
    ("Throttle", actions[:, 0]),
    ("Brake", actions[:, 1]),
    ("Steering", actions[:, 2])
]

for ax, (name, values) in zip(axes[:3], signals):
    ax.plot(time_s, values, marker="o", markersize=3)
    ax.set_ylabel(name)
    ax.set_title(f"{name} over time")
    ax.grid(alpha=0.3)

# Visualize camera-to-CAN timestamp differences.
axes[3].plot(time_s, gap_vm_ms, label="Throttle/Brake gap")
axes[3].plot(time_s, gap_steer_ms, label="Steering gap")
axes[3].axhline(
    250, color="red", linestyle="--",
    label="250 ms quality threshold"
)

# Highlight frames flagged by the alignment check.
flagged = ~action_valid

axes[3].scatter(
    time_s[flagged],
    gap_vm_ms[flagged],
    color="red",
    s=60,
    zorder=5,
    label="Flagged frames"
)

axes[3].set_ylabel("Gap (ms)")
axes[3].set_xlabel("Time from scene start (seconds)")
axes[3].set_title("Camera-to-CAN alignment gaps")
axes[3].legend()
axes[3].grid(alpha=0.3)

plt.suptitle(f"Driving Actions: {scene_name}", fontsize=15)
plt.tight_layout()
plt.show()

print("Total keyframes:", len(actions))
print("Flagged frames:", flagged.sum())
print("Action order: [throttle, brake, steer]")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

scene_names = sorted([
    f.replace(".npz", "")
    for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".npz")
])

assert len(scene_names) == 10, "Expected 10 processed scenes"

fig, axes = plt.subplots(
    len(scene_names), 3,
    figsize=(18, 24),
    squeeze=False
)

action_names = ["Throttle", "Brake", "Steering"]

total_frames = 0
total_flagged = 0

for row, scene_name in enumerate(scene_names):

    file_path = os.path.join(OUTPUT_DIR, f"{scene_name}.npz")

    with np.load(file_path) as data:
        actions = data["actions"]
        timestamps = data["timestamps"]
        valid = data["action_valid"]

    time_s = (timestamps - timestamps[0]) / 1e6
    flagged = ~valid

    total_frames += len(actions)
    total_flagged += int(flagged.sum())

    for col in range(3):
        ax = axes[row, col]

        ax.plot(
            time_s,
            actions[:, col],
            marker=".",
            markersize=3,
            linewidth=1.2
        )

        # Mark frames with large timestamp alignment gaps.
        if flagged.any():
            ax.scatter(
                time_s[flagged],
                actions[flagged, col],
                color="red",
                s=28,
                zorder=3
            )

        if row == 0:
            ax.set_title(action_names[col], fontsize=14)

        if col == 0:
            ax.set_ylabel(scene_name, fontsize=10)

        if row == len(scene_names) - 1:
            ax.set_xlabel("Time (seconds)")

        ax.grid(alpha=0.3)

plt.suptitle(
    "Driving Actions Across All 10 nuScenes Mini Scenes",
    fontsize=17,
    y=1.005
)

plt.tight_layout()
plt.show()

print("Scenes checked:", len(scene_names))
print("Total keyframes:", total_frames)
print("Flagged keyframes:", total_flagged)
print("Red markers indicate flagged timestamp alignments.")

assert total_frames == 404
assert total_flagged == 7

In [ ]:
import os
import numpy as np
import pandas as pd

scene_names = sorted(
    f[:-4] for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".npz")
)

def get_nearest_indices(source_times, target_times):
    pos = np.searchsorted(source_times, target_times)

    left = np.clip(pos - 1, 0, len(source_times) - 1)
    right = np.clip(pos, 0, len(source_times) - 1)

    choose_right = (
        np.abs(source_times[right] - target_times)
        < np.abs(source_times[left] - target_times)
    )

    return np.where(choose_right, right, left)


total_checked = 0
all_passed = True

for name in scene_names:

    with np.load(os.path.join(OUTPUT_DIR, f"{name}.npz")) as saved:
        timestamps = saved["timestamps"]
        actions = saved["actions"]
        vm_gaps = saved["gap_vm_ms"]
        steer_gaps = saved["gap_steer_ms"]

    vm = pd.DataFrame(
        can.get_messages(name, "vehicle_monitor")
    ).sort_values("utime")

    sa = pd.DataFrame(
        can.get_messages(name, "steeranglefeedback")
    ).sort_values("utime")

    vm_times = vm["utime"].to_numpy(dtype=np.int64)
    sa_times = sa["utime"].to_numpy(dtype=np.int64)

    vm_idx = get_nearest_indices(vm_times, timestamps)
    sa_idx = get_nearest_indices(sa_times, timestamps)

    expected_actions = np.column_stack([
        vm["throttle"].to_numpy()[vm_idx],
        vm["brake"].to_numpy()[vm_idx],
        sa["value"].to_numpy()[sa_idx]
    ]).astype(np.float32)

    expected_vm_gap = (
        np.abs(timestamps - vm_times[vm_idx]) / 1000
    )

    expected_steer_gap = (
        np.abs(timestamps - sa_times[sa_idx]) / 1000
    )

    action_check = np.allclose(
        actions, expected_actions, atol=1e-5, rtol=1e-5
    )

    gap_check = (
        np.allclose(vm_gaps, expected_vm_gap, atol=1e-3)
        and np.allclose(steer_gaps, expected_steer_gap, atol=1e-3)
    )

    passed = action_check and gap_check
    all_passed = all_passed and passed

    total_checked += len(timestamps)

    print(
        f"{name}: {'PASS' if passed else 'CHECK'} | "
        f"Actions: {action_check} | Gaps: {gap_check}"
    )

print("\nTotal scenes:", len(scene_names))
print("Total keyframes checked:", total_checked)
print("Final verification:", "PASSED" if all_passed else "NEEDS REVIEW")


## 12 Hz Camera Sweep Data — Initial Inspection

The initial preprocessing used 2 Hz camera keyframes, resulting in 404 frames across 10 nuScenes Mini scenes.

To increase the available training data, the intermediate camera sweep frames are now being examined.

The same three forward-facing cameras are retained:
- CAM_FRONT_LEFT
- CAM_FRONT
- CAM_FRONT_RIGHT

The target resolution remains 72 × 128, with all three RGB images stacked into a 9-channel representation.

This inspection checks the available camera sweep samples, their timestamps, and the number of non-keyframe images before extraction.


In [ ]:
import pandas as pd
import numpy as np

sweep_results = []

for scene in nusc.scene:

    scene_name = scene["name"]
    first_sample = nusc.get(
        "sample", scene["first_sample_token"]
    )

    for cam in CAMS:

        # Start from the first keyframe for this camera
        sd = nusc.get(
            "sample_data", first_sample["data"][cam]
        )

        # Move backwards to the first available sample_data
        while sd["prev"]:
            sd = nusc.get("sample_data", sd["prev"])

        records = []

        # Traverse the complete camera sample_data sequence
        while sd:
            records.append(sd)
            if not sd["next"]:
                break
            sd = nusc.get("sample_data", sd["next"])

        timestamps = np.array(
            [r["timestamp"] for r in records],
            dtype=np.int64
        )

        intervals_ms = np.diff(timestamps) / 1000

        sweep_results.append({
            "scene": scene_name,
            "camera": cam,
            "total_frames": len(records),
            "keyframes": sum(r["is_key_frame"] for r in records),
            "sweep_frames": sum(not r["is_key_frame"] for r in records),
            "median_interval_ms": (
                np.median(intervals_ms)
                if len(intervals_ms) else np.nan
            )
        })

sweep_df = pd.DataFrame(sweep_results)

display(sweep_df)

print("\nTotal camera records:", sweep_df["total_frames"].sum())
print("Total keyframes:", sweep_df["keyframes"].sum())
print("Total intermediate frames:", sweep_df["sweep_frames"].sum())


### Camera Sweep Frequency Analysis

The initial inspection identified 5,812 intermediate camera frames across the 10 nuScenes Mini scenes.

The median timestamp interval is approximately 100 ms across all three selected cameras. This corresponds to an observed sampling frequency of approximately 10 Hz.

Before extracting the additional frames, the timestamp intervals are examined in more detail to understand the available camera sampling rate and identify any irregular intervals.


In [ ]:
# Examine the observed sampling frequency for each camera

frequency_df = sweep_df.copy()

frequency_df["observed_frequency_hz"] = (
    1000 / frequency_df["median_interval_ms"]
)

display(
    frequency_df[
        [
            "scene",
            "camera",
            "median_interval_ms",
            "observed_frequency_hz",
            "sweep_frames"
        ]
    ].round(2)
)

print(
    "\nOverall median frequency:",
    round(
        1000 / frequency_df["median_interval_ms"].median(),
        2
    ),
    "Hz"
)

### Camera Sweep Timestamp Verification

Before extracting the additional camera sweep frames, I checked
the timestamp intervals for the three forward-facing cameras
across all 10 nuScenes Mini scenes.

This helps determine the actual camera sampling frequency
and whether the available frames support the 12 Hz target.

In [ ]:
import numpy as np
import pandas as pd

cameras = [
    "CAM_FRONT_LEFT",
    "CAM_FRONT",
    "CAM_FRONT_RIGHT"
]

results = []

for camera in cameras:

    intervals_ms = []
    frame_count = 0

    for scene in nusc.scene:

        # Collect keyframe and sweep timestamps for this scene.
        scene_samples = []
        token = scene["first_sample_token"]

        while token:
            sample = nusc.get("sample", token)
            scene_samples.append(sample)
            token = sample["next"]

        timestamps = []

        for sample in scene_samples:

            sd = nusc.get(
                "sample_data",
                sample["data"][camera]
            )

            # Follow the camera sample_data chain backward.
            while sd:
                timestamps.append(sd["timestamp"])

                if not sd["prev"]:
                    break

                sd = nusc.get("sample_data", sd["prev"])

        # Remove repeated records from overlapping chains.
        timestamps = np.array(
            sorted(set(timestamps)),
            dtype=np.int64
        )

        frame_count += len(timestamps)

        if len(timestamps) > 1:
            gaps = np.diff(timestamps) / 1000
            intervals_ms.extend(gaps.tolist())

    intervals_ms = np.array(intervals_ms)

    median_gap = np.median(intervals_ms)
    frequency = 1000 / median_gap

    results.append({
        "Camera": camera,
        "Frames": frame_count,
        "Median interval (ms)": round(median_gap, 2),
        "Estimated frequency (Hz)": round(frequency, 2),
        "Min interval (ms)": round(np.min(intervals_ms), 2),
        "Max interval (ms)": round(np.max(intervals_ms), 2)
    })

frequency_df = pd.DataFrame(results)

print("CAMERA SWEEP FREQUENCY CHECK")
print(frequency_df.to_string(index=False))

print("\nTarget frequency: 12 Hz")
print("Target interval: 83.33 ms")

### Camera Sweep Synchronization Check

The camera sweep timestamps showed a median frequency of 10 Hz.
I checked how closely the front-left and front-right camera frames
align with the front camera before combining them for training.

In [ ]:
import numpy as np
import pandas as pd

CAMS = ["CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT"]

sync_results = []

for scene in nusc.scene:
    scene_name = scene["name"]
    camera_times = {}

    first_sample = nusc.get("sample", scene["first_sample_token"])
    last_sample = nusc.get("sample", scene["last_sample_token"])

    for cam in CAMS:
        first_sd = nusc.get(
            "sample_data", first_sample["data"][cam]
        )
        last_sd = nusc.get(
            "sample_data", last_sample["data"][cam]
        )

        # Start at the earliest available sweep for this scene.
        sd = first_sd
        while sd["prev"]:
            previous = nusc.get("sample_data", sd["prev"])

            if previous["timestamp"] < first_sample["timestamp"] - 500000:
                break

            sd = previous

        timestamps = []

        # Follow the camera frames in chronological order.
        while True:
            timestamps.append(sd["timestamp"])

            if sd["token"] == last_sd["token"]:
                break

            if not sd["next"]:
                break

            sd = nusc.get("sample_data", sd["next"])

        camera_times[cam] = np.array(
            sorted(set(timestamps)), dtype=np.int64
        )

    front_times = camera_times["CAM_FRONT"]

    for cam in ["CAM_FRONT_LEFT", "CAM_FRONT_RIGHT"]:
        other_times = camera_times[cam]

        positions = np.searchsorted(other_times, front_times)
        right = np.clip(positions, 0, len(other_times) - 1)
        left = np.clip(positions - 1, 0, len(other_times) - 1)

        differences = np.minimum(
            np.abs(other_times[right] - front_times),
            np.abs(other_times[left] - front_times)
        ) / 1000.0

        sync_results.append({
            "Scene": scene_name,
            "Camera": cam,
            "Front frames": len(front_times),
            "Median gap (ms)": round(float(np.median(differences)), 2),
            "Max gap (ms)": round(float(np.max(differences)), 2),
            "Within 50 ms (%)": round(
                float(np.mean(differences <= 50) * 100), 2
            )
        })

sync_df = pd.DataFrame(sync_results)

print("CAMERA SYNCHRONIZATION CHECK")
print(sync_df.to_string(index=False))

print("\nOVERALL SUMMARY")
print(
    sync_df.groupby("Camera")[
        ["Median gap (ms)", "Max gap (ms)", "Within 50 ms (%)"]
    ].mean().round(2)
)

### Building Synchronized Camera Sweep Sequences

I used CAM_FRONT as the reference camera and matched the
nearest frames from CAM_FRONT_LEFT and CAM_FRONT_RIGHT.

Matches within 50 ms were considered synchronized.
Frames outside this threshold were flagged before extraction.

The original camera timestamps are preserved, and the
existing keyframe dataset is kept unchanged.

In [ ]:
import numpy as np
import pandas as pd

CAMS = ["CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT"]
SYNC_THRESHOLD_US = 50_000

sweep_matches = {}
sweep_summary = []

def get_scene_camera_records(scene, camera):
    first_sample = nusc.get("sample", scene["first_sample_token"])
    last_sample = nusc.get("sample", scene["last_sample_token"])

    first_sd = nusc.get(
        "sample_data", first_sample["data"][camera]
    )
    last_sd = nusc.get(
        "sample_data", last_sample["data"][camera]
    )

    records = []
    sd = first_sd

    # Include sweeps immediately preceding the first keyframe.
    while sd["prev"]:
        previous = nusc.get("sample_data", sd["prev"])

        if previous["timestamp"] < first_sample["timestamp"] - 500_000:
            break

        sd = previous

    # Collect frames up to the last scene keyframe.
    while True:
        records.append({
            "timestamp": sd["timestamp"],
            "token": sd["token"],
            "filename": sd["filename"]
        })

        if sd["token"] == last_sd["token"] or not sd["next"]:
            break

        sd = nusc.get("sample_data", sd["next"])

    return sorted(records, key=lambda x: x["timestamp"])


def nearest_record(records, timestamp):
    times = np.array(
        [r["timestamp"] for r in records],
        dtype=np.int64
    )

    pos = np.searchsorted(times, timestamp)

    candidates = [
        i for i in [pos - 1, pos]
        if 0 <= i < len(records)
    ]

    best = min(
        candidates,
        key=lambda i: abs(int(times[i]) - int(timestamp))
    )

    gap_us = abs(int(times[best]) - int(timestamp))

    return records[best], gap_us


for scene in nusc.scene:
    name = scene["name"]

    camera_records = {
        cam: get_scene_camera_records(scene, cam)
        for cam in CAMS
    }

    matches = []

    for front in camera_records["CAM_FRONT"]:
        timestamp = front["timestamp"]

        left, left_gap = nearest_record(
            camera_records["CAM_FRONT_LEFT"], timestamp
        )

        right, right_gap = nearest_record(
            camera_records["CAM_FRONT_RIGHT"], timestamp
        )

        valid = (
            left_gap <= SYNC_THRESHOLD_US
            and right_gap <= SYNC_THRESHOLD_US
        )

        matches.append({
            "timestamp": timestamp,
            "front": front["filename"],
            "left": left["filename"],
            "right": right["filename"],
            "left_gap_ms": left_gap / 1000,
            "right_gap_ms": right_gap / 1000,
            "sync_valid": valid
        })

    sweep_matches[name] = matches

    valid_count = sum(m["sync_valid"] for m in matches)

    sweep_summary.append({
        "Scene": name,
        "Front frames": len(matches),
        "Valid triplets": valid_count,
        "Flagged": len(matches) - valid_count
    })

summary_df = pd.DataFrame(sweep_summary)

print("SYNCHRONIZED CAMERA SWEEP SUMMARY")
print(summary_df.to_string(index=False))

print("\nTOTAL")
print("Front reference frames:", summary_df["Front frames"].sum())
print("Valid triplets:", summary_df["Valid triplets"].sum())
print("Flagged triplets:", summary_df["Flagged"].sum())

In [ ]:
import os
import numpy as np
from PIL import Image

# Use the first valid triplet from scene-0061.
test_match = next(
    m for m in sweep_matches["scene-0061"]
    if m["sync_valid"]
)

camera_order = ["left", "front", "right"]
images = []

for camera in camera_order:
    image_path = os.path.join(
        nusc.dataroot,
        test_match[camera]
    )

    assert os.path.isfile(image_path), (
        f"Missing image: {image_path}"
    )

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img = img.resize((128, 72), Image.Resampling.BILINEAR)
        images.append(np.array(img, dtype=np.uint8))

# Combine three RGB images into nine channels.
stacked_frame = np.concatenate(images, axis=2)

assert stacked_frame.shape == (72, 128, 9)
assert stacked_frame.dtype == np.uint8

print("Scene: scene-0061")
print("Image shape:", stacked_frame.shape)
print("Data type:", stacked_frame.dtype)
print("Left camera gap:", test_match["left_gap_ms"], "ms")
print("Right camera gap:", test_match["right_gap_ms"], "ms")
print("Image extraction test: PASSED")

### Full Camera Sweep Image Extraction

After testing one synchronized camera triplet, I extracted
the valid camera sweep frames from all 10 scenes.

Each frame combines the front-left, front, and front-right
camera images into a 9-channel array at 72 × 128 resolution.

Only camera triplets within the 50 ms synchronization
threshold are included.

In [ ]:
import os
import numpy as np
from PIL import Image

SWEEP_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR, "camera_sweeps"
)

os.makedirs(SWEEP_OUTPUT_DIR, exist_ok=True)

total_extracted = 0

for scene_name, matches in sweep_matches.items():

    valid_matches = [
        m for m in matches if m["sync_valid"]
    ]

    frames = []
    timestamps = []

    for match in valid_matches:

        images = []

        for camera in ["left", "front", "right"]:

            image_path = os.path.join(
                nusc.dataroot, match[camera]
            )

            with Image.open(image_path) as img:
                img = img.convert("RGB")
                img = img.resize(
                    (128, 72),
                    Image.Resampling.BILINEAR
                )

                images.append(
                    np.array(img, dtype=np.uint8)
                )

        stacked = np.concatenate(images, axis=2)

        # Convert HWC to CHW for model input.
        stacked = np.transpose(stacked, (2, 0, 1))

        assert stacked.shape == (9, 72, 128)

        frames.append(stacked)
        timestamps.append(match["timestamp"])

    frames = np.stack(frames)
    timestamps = np.array(timestamps, dtype=np.int64)

    save_path = os.path.join(
        SWEEP_OUTPUT_DIR, f"{scene_name}_sweeps.npz"
    )

    np.savez_compressed(
        save_path,
        frames=frames,
        timestamps=timestamps
    )

    total_extracted += len(frames)

    print(
        f"{scene_name}: {len(frames)} frames saved"
    )

print("\nTotal extracted:", total_extracted)

assert total_extracted == 2313

print("Camera sweep image extraction: PASSED")

### CAN-bus Alignment for Camera Sweeps

I matched the extracted camera sweep timestamps with the
vehicle's throttle, brake, and steering measurements.

I used nearest-timestamp matching, following the same method
used for the original keyframe dataset.

Frames with CAN-bus alignment gaps greater than 250 ms
were flagged for review.

In [ ]:
import os
import numpy as np
import pandas as pd

SWEEP_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "camera_sweeps")
ALIGNED_SWEEP_DIR = os.path.join(OUTPUT_DIR, "camera_sweeps_aligned")

os.makedirs(ALIGNED_SWEEP_DIR, exist_ok=True)

total_frames = 0
total_flagged = 0

for scene in nusc.scene:

    name = scene["name"]

    input_path = os.path.join(
        SWEEP_OUTPUT_DIR, f"{name}_sweeps.npz"
    )

    assert os.path.isfile(input_path), f"Missing sweep file: {name}"

    # Load the saved camera frames and timestamps.
    with np.load(input_path) as data:
        frames = data["frames"]
        timestamps = data["timestamps"].astype(np.int64)

    # Read the same CAN-bus signals used for keyframes.
    vm = pd.DataFrame(
        can.get_messages(name, "vehicle_monitor")
    )[["utime", "throttle", "brake"]].sort_values("utime")

    sa = pd.DataFrame(
        can.get_messages(name, "steeranglefeedback")
    )[["utime", "value"]].sort_values("utime")

    vm = vm.rename(columns={"utime": "vm_time"})

    sa = sa.rename(columns={
        "utime": "steer_time",
        "value": "steer"
    })

    assert not vm.empty, f"No vehicle monitor data: {name}"
    assert not sa.empty, f"No steering data: {name}"

    # Match each sweep timestamp to the nearest CAN reading.
    keys = pd.DataFrame({
        "timestamp": timestamps
    }).sort_values("timestamp")

    matched = pd.merge_asof(
        keys,
        vm,
        left_on="timestamp",
        right_on="vm_time",
        direction="nearest"
    )

    matched = pd.merge_asof(
        matched,
        sa,
        left_on="timestamp",
        right_on="steer_time",
        direction="nearest"
    )

    # Calculate timestamp differences in milliseconds.
    matched["gap_vm_ms"] = (
        (matched["timestamp"] - matched["vm_time"]).abs() / 1000
    )

    matched["gap_steer_ms"] = (
        (matched["timestamp"] - matched["steer_time"]).abs() / 1000
    )

    matched["action_valid"] = (
        (matched["gap_vm_ms"] <= 250) &
        (matched["gap_steer_ms"] <= 250)
    )

    # Keep actions in the same order as the keyframe dataset.
    actions = matched[
        ["throttle", "brake", "steer"]
    ].to_numpy(dtype=np.float32)

    action_valid = matched["action_valid"].to_numpy(dtype=bool)

    assert frames.shape == (len(timestamps), 9, 72, 128)
    assert actions.shape == (len(timestamps), 3)
    assert np.isfinite(actions).all()
    assert np.array_equal(
        matched["timestamp"].to_numpy(dtype=np.int64),
        timestamps
    )

    # Save aligned sweeps separately from the original files.
    output_path = os.path.join(
        ALIGNED_SWEEP_DIR, f"{name}_sweeps.npz"
    )

    np.savez_compressed(
        output_path,
        frames=frames,
        timestamps=timestamps,
        actions=actions,
        gap_vm_ms=matched["gap_vm_ms"].to_numpy(dtype=np.float32),
        gap_steer_ms=matched["gap_steer_ms"].to_numpy(dtype=np.float32),
        action_valid=action_valid
    )

    flagged = int((~action_valid).sum())

    total_frames += len(timestamps)
    total_flagged += flagged

    print(
        f"{name}: {len(timestamps)} frames | "
        f"{flagged} action flagged"
    )

print("\nTotal aligned sweep frames:", total_frames)
print("Total action flagged:", total_flagged)

assert total_frames == 2313

print("CAN-bus sweep alignment: PASSED")
print("Saved to:", ALIGNED_SWEEP_DIR)

In [ ]:
import os
import json
import numpy as np

OUTPUT_DIR = "/content/drive/MyDrive/DriveJEPA/nuscenes_processed"

ALIGNED_SWEEP_DIR = os.path.join(
    OUTPUT_DIR, "camera_sweeps_aligned"
)

with open(os.path.join(OUTPUT_DIR, "scene_split.json"), "r") as f:
    split = json.load(f)

expected_counts = {
    "scene-0061": 213,
    "scene-0103": 226,
    "scene-0553": 236,
    "scene-0655": 234,
    "scene-0757": 236,
    "scene-0796": 232,
    "scene-0916": 239,
    "scene-1077": 236,
    "scene-1094": 230,
    "scene-1100": 231
}

assert len(split) == 10
assert set(split) == set(expected_counts)
assert sum(v["split"] == "train" for v in split.values()) == 7
assert sum(v["split"] == "val" for v in split.values()) == 3

total_frames = 0
total_flagged = 0

for group in ["train", "val"]:

    group_frames = 0
    group_flagged = 0

    print(f"\n{group.upper()} SWEEP DATASET")

    for scene_name, info in split.items():

        if info["split"] != group:
            continue

        file_path = os.path.join(
            ALIGNED_SWEEP_DIR,
            f"{scene_name}_sweeps.npz"
        )

        assert os.path.isfile(file_path), scene_name

        with np.load(file_path) as data:

            n = len(data["frames"])

            assert n == expected_counts[scene_name]
            assert data["frames"].shape == (n, 9, 72, 128)
            assert data["frames"].dtype == np.uint8
            assert data["actions"].shape == (n, 3)
            assert len(data["timestamps"]) == n
            assert len(data["action_valid"]) == n
            assert len(data["gap_vm_ms"]) == n
            assert len(data["gap_steer_ms"]) == n

            assert np.isfinite(data["actions"]).all()
            assert np.isfinite(data["gap_vm_ms"]).all()
            assert np.isfinite(data["gap_steer_ms"]).all()

            timestamps = data["timestamps"]
            assert np.all(np.diff(timestamps) > 0)

            expected_valid = (
                (data["gap_vm_ms"] <= 250) &
                (data["gap_steer_ms"] <= 250)
            )

            assert np.array_equal(
                data["action_valid"],
                expected_valid
            )

            flagged = int((~data["action_valid"]).sum())

            group_frames += n
            group_flagged += flagged

            print(
                f"{scene_name}: {n} frames | "
                f"{flagged} flagged"
            )

    total_frames += group_frames
    total_flagged += group_flagged

    print(f"{group} total: {group_frames}")
    print(f"{group} flagged: {group_flagged}")

assert total_frames == 2313
assert total_flagged == 35

print("\nTotal frames:", total_frames)
print("Total flagged:", total_flagged)
print("Usable frames:", total_frames - total_flagged)
print("FINAL SWEEP DATASET VERIFICATION: PASSED")